# eCooking mini-grid model — v5 Colab driver

All code now lives in the repository; this notebook only runs it.
v4's `%%writefile` cells are retired, so the notebook and the paper's code
cannot drift apart.

1. Install: Colab runs Python 3.13, but the pinned model stack (RAMP 0.5.0,
   numpy 1.26) needs Python 3.11, so `uv` builds a separate Python 3.11
   environment and every model command runs with it (`{PY}`). No restart needed.
2. Google Drive keeps the results and the weather data, so a runtime reset loses nothing
3. `pipeline.py all --resume` regenerates every result, table and figure, and
   carries on from the last finished step if it is run again
4. Tests, and the manuscript-vs-results check

**If Colab stops or resets part-way:** run cells 1 and 2 again, then go
straight to the experiments cell. Finished steps are skipped.


In [ ]:
!git clone -q https://github.com/chilalaabubakar/EDA.git /content/ecook 2>/dev/null || (cd /content/ecook && git pull -q)
%cd /content/ecook
!git log --oneline -1

# Colab's own Python is 3.13; the pinned stack needs 3.11. uv downloads a
# Python 3.11 and installs the exact pinned versions into /content/py311.
import os
# Colab sets MPLBACKEND to its inline plotting backend, which the Python 3.11
# environment does not have: run every model command headless instead.
os.environ["MPLBACKEND"] = "Agg"
PY = "/content/py311/bin/python"
if not os.path.exists(PY):
    !pip install -q uv
    !uv venv -q --python 3.11 /content/py311
!uv pip install -q --python {PY} -r requirements.txt
!{PY} -c "import sys, numpy, ramp, numpy_financial, docx; print('model environment ready: Python', sys.version.split()[0], '| numpy', numpy.__version__)"


In [ ]:
%cd /content/ecook
import os, shutil
# Colab sets MPLBACKEND to its inline plotting backend, which the Python 3.11
# environment does not have: run every model command headless instead.
os.environ["MPLBACKEND"] = "Agg"
PY = "/content/py311/bin/python"
assert os.path.exists(PY), "run cell 1 first"
from google.colab import drive
drive.mount("/content/drive")

# Everything worth keeping lives on Drive, so a runtime reset loses nothing.
KEEP = "/content/drive/MyDrive/ecook_run"
os.makedirs(f"{KEEP}/results", exist_ok=True)
os.makedirs(f"{KEEP}/resource", exist_ok=True)
if not os.path.islink("results"):
    shutil.rmtree("results", ignore_errors=True)
    os.symlink(f"{KEEP}/results", "results")
print("results ->", os.path.realpath("results"))

# NASA POWER 2005-2024, both sites, local solar time: fetched once, then kept on Drive
SITES = ("rwanda", "kenya")
for s in SITES:
    kept, here = f"{KEEP}/resource/resource_{s}.csv", f"data/processed/resource_{s}.csv"
    if os.path.exists(kept) and not os.path.exists(here):
        shutil.copy(kept, here)
if all(os.path.exists(f"data/processed/resource_{s}.csv") for s in SITES):
    print("weather data restored from Drive")
else:
    !{PY} src/pipeline.py resource
    for s in SITES:
        shutil.copy(f"data/processed/resource_{s}.csv", f"{KEEP}/resource/")


## Survey microdata (needed only to re-derive the MTF inputs)

The derived appliance sets are committed in `src/final_cfg.py`, so the model
runs without this step. Rwanda MTF 2022 is licensed: download
`RWA_2022_MTF_v01_M_CSV.zip` from catalog 6429 and put it in Drive `/mtf/`
(see `docs/DATA_ACCESS.md`). Never commit it.

In [ ]:
USE_DRIVE = True
import glob, zipfile, os
os.makedirs("data/raw", exist_ok=True)
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    for z in glob.glob("/content/drive/MyDrive/mtf/*.zip"):
        zipfile.ZipFile(z).extractall("data/raw"); print("extracted", os.path.basename(z))
if os.path.exists("data/raw/household_survey_data/SECTION_EF1.csv"):
    !{PY} src/pipeline.py mtf
else:
    print("microdata absent - using the committed MTF-derived values")

In [ ]:
# Phases 1-4: sweep, bands, validation, viability, tables, expansion, sizing
# check, interannual, 24-seed ensemble, ToU sweep, ESMAP, operating subsidy,
# household bands, peaks, Ringiti, household cost, figures.
# --resume skips the steps already listed in results/.steps_done, so if the
# runtime resets, rerun cells 1-2 and this cell: it carries on where it stopped.
# To start again from scratch, delete MyDrive/ecook_run/results/.steps_done.
!{PY} src/pipeline.py all --resume


In [ ]:
!{PY} -m pytest -q
!cat results/tables/tables.md
!{PY} src/manuscript_tables.py --abstract

## Manuscript check

Upload the current draft; every table cell and headline number is compared
with `results/`. Mismatches list exactly what to change.

In [ ]:
from google.colab import files
up = files.upload()
os.makedirs("manuscript/private", exist_ok=True)
for name, data in up.items():
    open(f"manuscript/private/{name}", "wb").write(data)
    !{PY} src/manuscript_check.py "manuscript/private/{name}"
    !{PY} src/word_count.py "manuscript/private/{name}" --limit 5000

In [ ]:
# Commit results (not data, not the manuscript) back to the repo, if desired:
# !git add results data/processed && git commit -m "Results from Colab run" && git push
!zip -qr results.zip results && echo results.zip ready
files.download("results.zip")